# Chapter 46: Introductory Causal Inference

A synthetic NRG coaching study demonstrates confounding, propensity weighting, overlap, and balance.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.causal import difference_in_means,ipw_weights,weighted_effect,standardized_mean_difference
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(46);n=600
prior=rng.normal(100,22,n);tenure=rng.uniform(1,8,n)
logit=-4.2+.035*prior+.18*tenure;p=1/(1+np.exp(-logit));t=rng.binomial(1,p)
true_effect=8.0;y=35+.72*prior+1.5*tenure+true_effect*t+rng.normal(0,12,n)
print(f'Distributors={n}; coached={t.sum()}; true effect={true_effect:.1f}')


Distributors=600; coached=307; true effect=8.0


In [ ]:
raw=difference_in_means(y,t);model=LogisticRegression().fit(np.c_[prior,tenure],t);phat=model.predict_proba(np.c_[prior,tenure])[:,1];w=ipw_weights(t,phat);adjusted=weighted_effect(y,t,w)
print(f'Raw difference={raw:.2f}; weighted effect={adjusted:.2f}; max weight={w.max():.2f}')


Raw difference=14.65; weighted effect=6.45; max weight=8.61


In [ ]:
before=standardized_mean_difference(prior,t);after=standardized_mean_difference(prior,t,w)
print(f'Prior-sales SMD before={before:.3f}; after={after:.3f}')


Prior-sales SMD before=0.462; after=-0.036


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4));axes[0].hist(phat[t==0],bins=20,alpha=.6,label='not coached');axes[0].hist(phat[t==1],bins=20,alpha=.6,label='coached');axes[0].set(title='Propensity overlap',xlabel='Estimated propensity');axes[0].legend();axes[1].bar(['Raw','IPW'],[abs(before),abs(after)]);axes[1].axhline(.1,color='black',ls=':');axes[1].set(title='Prior-sales balance',ylabel='Absolute SMD');fig.tight_layout();plt.show()


## Interpretation

Weighting moves the estimate toward the simulated effect and improves measured balance. This does not prove exchangeability: an unmeasured confounder, poor treatment definition, interference, or unsupported population can still invalidate a causal interpretation.


In [ ]:
# Practice: add a baseline region variable and check its balance before and after weighting.
